[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Favioleiva/gpubma/blob/main/examples/Exact_BMA_Canonical_p30_Figures.ipynb)

# Exact BMA canonical p30 figures

Reproduce the publication suite from all **1,073,741,824 models** using committed, SHA-256-verified exact artifacts. This is the full-enumeration reference notebook; the separate BFG notebook performs budgeted discovery.

The default path evaluates **zero models** and needs no CUDA. Configuration: n=2,000, x1–x30, always-in w1/w2, g=2,000, beta-binomial(1,1), float64. The reference is frozen; exact global quantities are not inferred from a BFG subset.

Run All from the public checkout. In standalone Colab, the bootstrap obtains the same committed inputs after human publication of this tree. Network access is required for missing dependencies/downloads. Raw multi-gigabyte arrays are optional and are not publicly hosted by this candidate.

In [ ]:
from pathlib import Path
import hashlib, json, sys, subprocess, importlib.metadata
from urllib.request import urlopen

EXPECTED_FILES = {'benchmark/exact_p30/reference/REFERENCE_MANIFEST.json': '25425f071e4bb29063506709af284c9303f5ba180b3b231f0816b361f4a4e5b9', 'benchmark/exact_p30/reference/REFERENCE_MANIFEST.sha256': '0ebf5ece66c4ff872a803fd18a9e972fd6bb1681153263a7670d07fbd9224238', 'examples/exact_reference_figures.py': '60096dcba41bd201d653c79c273c953838c553794816adb534ce5dc749447583', 'examples/exact_reference_workflow.py': 'fac9f53df32da7056a8aa39f41d0e4ee9a083cee8e713c30950e22e0e1f8c78d'}
PUBLIC_RAW = "https://raw.githubusercontent.com/Favioleiva/gpubma/main/"
roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in roots if (p / "benchmark/exact_p30/reference/REFERENCE_MANIFEST.json").is_file()), None)
LOCAL_CHECKOUT = ROOT is not None
if ROOT is None:
    ROOT = Path.cwd() / "gpubma_exact_reference"
    ROOT.mkdir(exist_ok=True)
    def download_checked(relative, expected):
        target = ROOT / relative
        if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == expected:
            return
        with urlopen(PUBLIC_RAW + relative, timeout=90) as response:
            payload = response.read()
        if hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError("Public reference checksum mismatch: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
    for relative, digest in EXPECTED_FILES.items():
        download_checked(relative, digest)
    manifest = json.loads((ROOT / "benchmark/exact_p30/reference/REFERENCE_MANIFEST.json").read_text())
    for name, digest in manifest["files"].items():
        if Path(name).name != name:
            raise ValueError("Unsafe reference filename")
        download_checked("benchmark/exact_p30/reference/" + name, digest)
for relative, digest in EXPECTED_FILES.items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == digest, relative

try:
    installed = importlib.metadata.version("gpubma")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "0.3.0rc1":
    requirement = str(ROOT) + "[notebooks]" if LOCAL_CHECKOUT else "gpubma[notebooks] @ git+https://github.com/Favioleiva/gpubma.git"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", requirement])
missing = []
for module, distribution in [("matplotlib", "matplotlib>=3.7"), ("pandas", "pandas>=2.1"), ("scipy", "scipy>=1.11")]:
    if importlib.util.find_spec(module) is None:
        missing.append(distribution)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import gpubma
sys.path.insert(0, str(ROOT / "examples"))
from exact_reference_workflow import load_reference, export_tables
from exact_reference_figures import render_suite, FIGURE_CATALOG
from IPython.display import display, Markdown, Image
print("gpubma", importlib.metadata.version("gpubma"), "— exact reference ready")


In [ ]:
REGENERATE_FULL_ENUMERATION = False
OUTPUT_DIR = ROOT / "outputs" / "exact_p30"
REFERENCE_DIR = ROOT / "benchmark" / "exact_p30" / "reference"
if REGENERATE_FULL_ENUMERATION:
    raise RuntimeError("This evaluates all 1,073,741,824 models. Full regeneration requires the separately provisioned canonical CUDA enumerator and a fresh output location; this figure notebook loads the frozen reference only.")
metadata, exact_map, exact_truth = load_reference(REFERENCE_DIR)
print("Models represented:", f"{metadata['models']:,}")
print("Models evaluated by this notebook: 0")
print("Frozen RTX 3060 enumeration runtime: 379.812 s; throughput: 2,827,037 models/s")
display(exact_map, exact_truth)
catalog, validation = render_suite(REFERENCE_DIR, OUTPUT_DIR)
table_path = export_tables(REFERENCE_DIR, OUTPUT_DIR)
assert len(catalog) == 9
assert all(v["embedded_titles"] == v["embedded_captions"] == 0 for v in validation["figures"].values())


In [ ]:
display(Markdown('### Canonical p30 predictor geometry'))
display(Image(filename=str(OUTPUT_DIR / '00_predictor_correlogram.png')))
display(Markdown('**Notes:** Observed candidate correlations, n = 2,000; original metadata order. Controls w1, w2 excluded. Not a causal or posterior map. Blue labels: causal DGP predictors x1–x15; red labels: structural-zero proxies x16–x30.'))

In [ ]:
display(Markdown('### Full-enumeration model-space cloud'))
display(Image(filename=str(OUTPUT_DIR / '01_exhaustive_model_space_cloud.png')))
display(Markdown('**Notes:** All 1,073,741,824 models contribute once. Exact counts in 2,000 common score bins; no random thinning. Lines: exact support.'))

In [ ]:
display(Markdown('### Exact log BMA numerator across all reticula'))
display(Image(filename=str(OUTPUT_DIR / '02_exhaustive_reticular_ridgeline.png')))
display(Markdown('**Notes:** Full-shell histograms; Gaussian smoothing σ = 3 bins for display. Unit-peak ridge height is not posterior mass. Exact finite support; singleton spikes.'))

In [ ]:
display(Markdown('### Exact lower and upper reticular frontiers'))
display(Image(filename=str(OUTPUT_DIR / '03_exhaustive_reticular_frontiers.png')))
display(Markdown('**Notes:** Left: exact lower reticular frontier. Right: exact upper reticular frontier. Every shell is fully enumerated. The MAP and true model both have k=15, but their compositions differ by two predictor indicators. Short marker callouts are anchored at the exact scores.'))

In [ ]:
display(Markdown('### Exact global variable / model structure'))
display(Image(filename=str(OUTPUT_DIR / '04_exact_global_variable_map.png')))
display(Markdown('**Notes:** Widths use exact global PMP; inclusion sidebar sums all 2³⁰ models. Cell colors describe model-specific OLS signs. Always-in controls excluded.'))

In [ ]:
display(Markdown('### Exact top models: composition and global model probabilities'))
display(Image(filename=str(OUTPUT_DIR / '05_exact_top_models_composition_weights.png')))
display(Markdown('**Notes:** Shared row coordinates preserve rank / composition / weight alignment. Candidates ordered by exact global PIP; w1 and w2 always included.'))

In [ ]:
display(Markdown('### Exact global posterior model-size distribution'))
display(Image(filename=str(OUTPUT_DIR / '06_exact_global_model_size_distribution.png')))
display(Markdown('**Notes:** Candidate size excludes the intercept and always-in controls w1, w2. Posterior probabilities aggregate the entire model universe.'))

In [ ]:
display(Markdown('### Exact BMA coefficient structure: archived full-universe mixture'))
display(Image(filename=str(OUTPUT_DIR / '07_exact_global_coefficient_structure.png')))
display(Markdown('**Notes:** Verified A100 exact-mixture artifact reused on its 257-point grids. Curves condition on inclusion; exclusion atoms at zero omitted. Height is unit-peak. Rows read top to bottom: blue positive effects (largest first), red negative effects (most negative first), then gray predictors (PIP descending). Original colors and inclusion rule are unchanged.'))

In [ ]:
display(Markdown('### Causal generating model versus exact posterior champion'))
display(Image(filename=str(OUTPUT_DIR / '08_exact_true_model_vs_map.png')))
display(Markdown('**Notes:** Known synthetic truth is distinct from posterior optimality. Red outlines identify differing predictor indicators; controls w1, w2 are common. Hamming distance = 2; x15 is replaced by proxy x30 in the MAP.'))

In [ ]:
display(Markdown("### Exact top-five regression comparison"))
import pandas as pd
display(pd.read_csv(OUTPUT_DIR / "exact_top5_regression_diagnostics.csv"))
display(Markdown("**Notes:** The exported booktabs table includes a dedicated M* reference panel: x1–x15, size 15, exact rank 8, PMP 0.01881348830750715. MAP is x1–x14+x30, size 15, PMP 0.4715187997893168. The substitution x15 → x30 changes two binary inclusion indicators. Conventional OLS standard errors are not posterior standard deviations."))
print("LaTeX table:", table_path.relative_to(ROOT).as_posix())
print("Exports:", OUTPUT_DIR.relative_to(ROOT).as_posix())
print("\n".join(p.name for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()))
